# Практическая работа №4. Часть 2Р
## Распознавание речи: CTC, Whisper, Vosk, метрики WER и CER

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ExPl0i/AI-10-Speech-ASR/blob/main/%D0%9F%D0%A0-%D0%A0%D0%B0%D1%81%D0%BF%D0%BE%D0%B7%D0%BD%D0%B0%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5%20%D1%80%D0%B5%D1%87%D0%B8.ipynb)

**Цель:** понять, как модель переводит длинную запись в текст, измерить качество и скорость готовых распознавателей и выбрать модель для голосового помощника водителя на слабом компьютере. В части 1З сеть различала шесть команд; теперь нужно распознавать свободные фразы: «построй маршрут до улицы Весенней, дом двадцать пять».

> **Правило части.** Декодирование CTC, расстояние редактирования, WER и CER, коэффициент реального времени и разбор ошибок пишем сами. Готовые распознаватели (Vosk, Whisper) и библиотеку `jiwer` используем для распознавания и сверки.

**Модели.**
- **Vosk** `vosk-model-small-ru-0.22` – компактная автономная модель для русского языка (45 МБ, лицензия Apache 2.0, alphacephei.com/vosk). Работает на процессоре быстрее реального времени и без интернета.
- **Whisper** tiny, base, small (OpenAI, лицензия MIT) – трансформер с кодировщиком и декодером, обученный на 680 тысячах часов речи на многих языках; загружается через `transformers` с Hugging Face.

**Данные** – выбор в первой ячейке (`DATA_SOURCE`).
- `'группа'` – ваши длинные фразы водителя из домашнего задания Т2.Б1.5: преподаватель переводит их скриптом `prepare_phrases.py` в папку `data/phrases/<ученик>/<имя>.wav`, рядом с каждой записью лежит файл `<имя>.txt` с точной расшифровкой. Записи – персональные данные, они хранятся только на школьном носителе.
- `'запасной'` – 20 фраз водителя (адреса улиц Кемерова, числа, команды), озвученные синтезатором речи Piper двумя голосами: `irina` и `denis`. Их делает `prepare_data.py` или ячейка загрузки данных (нужна библиотека `piper-tts`). Синтезированная речь чище живой, поэтому ошибок на ней меньше.

К каждой записи добавляется ее версия с шумом двигателя (отношение сигнал/шум 5 дБ), чтобы увидеть, как шум ломает распознавание.

**Два режима** выбираются в первой ячейке.
- `'легкий'` – компьютер в аудитории: 10 фраз одним голосом и их версии с шумом, модели Vosk, Whisper tiny и base на процессоре.
- `'полный'` – Colab с видеокартой: все фразы обоими голосами, плюс Whisper small на видеокарте.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом, подсказка в комментарии `# <--`. Проверки на маленьких примерах должны совпасть с тем, что стоит после слова «ожидается», точно. Тексты распознавателей, ошибки и время зависят от записей и компьютера, поэтому для них ожидания описаны словами.

## Как запись становится текстом

**1. От команды к свободной речи.** В части 1З каждая запись длилась секунду и относилась к одному из шести классов. Во фразе слов сколько угодно, запись длится несколько секунд: 300 кадров мел-спектрограммы на 3 секунды, а букв в тексте около 40. Длины звука и текста не совпадают, и в обучающих данных никто не размечал, какой кусок записи соответствует какой букве: есть только запись и ее текст целиком. Нужен способ учить модель без такой разметки.

**2. Функция потерь CTC** (англ. Connectionist Temporal Classification, Грейвс и соавторы, 2006). Модель выдает для каждого кадра распределение вероятностей по алфавиту, к которому добавлен **пустой символ** (будем писать его как `-`). Из последовательности символов по кадрам (пути) текст получается по правилу: **сначала склеить подряд идущие одинаковые символы, потом удалить пустые**. Например, `--сс-т-оо--п-` → `стоп`. Пустой символ нужен, чтобы записать двойную букву: `кк-а-сс-с-аа` → `касса`, а `кассса` → `каса`.

Вероятность текста – сумма вероятностей **всех** путей, которые дают этот текст; вероятность пути – произведение вероятностей его символов по кадрам (кадры считаются независимыми при заданном звуке). Потери CTC – минус логарифм вероятности правильного текста. Путей экспоненциально много, но сумму считают динамическим программированием за время, пропорциональное (число кадров) × (длина текста), – это делает `torch.nn.functional.ctc_loss`.

**Жадное декодирование**: в каждом кадре взять самый вероятный символ и применить правило склеивания. Это быстро, но находит самый вероятный **путь**, а не самый вероятный **текст**: при двух кадрах с вероятностями пустого символа 0,6 и буквы «а» 0,4 жадный путь `--` дает пустой текст с вероятностью 0,36, а текст «а» получается тремя путями `а-`, `-а`, `аа` с общей вероятностью 0,64. **Лучевой поиск** (англ. beam search) хранит несколько лучших вариантов текста и складывает вероятности путей – это задание повышенной сложности.

**3. Whisper** (Рэдфорд и соавторы, 2023) – трансформер с кодировщиком и декодером (семейство из Т2.Б1.4):
- вход – логарифмическая мел-спектрограмма **ровно 30 секунд** (80 мел-полос, окно 25 мс, шаг 10 мс, 3000 кадров); короткую запись дополняют тишиной, длинную режут на куски по 30 секунд;
- кодировщик: две одномерные свертки (вторая с шагом 2 – 1500 позиций) и блоки трансформера;
- декодер пишет текст токенами BPE (словарь 51 865 токенов) по одному, как GPT, и через перекрестное внимание смотрит на нужный кусок звука;
- **служебные токены** в начале задают задачу: `<|startoftranscript|><|ru|><|transcribe|><|notimestamps|>` – «распознай русскую речь без отметок времени»; вместо `<|transcribe|>` можно поставить `<|translate|>` – перевод на английский;
- обучен на 680 тысячах часов речи из интернета с расшифровками, поэтому устойчив к шуму и акцентам и сам расставляет знаки препинания и заглавные буквы;
- размеры: tiny – 38 млн параметров, base – 73 млн, small – 242 млн (151, 290 и 967 МБ в 32-битном формате); есть еще medium и large.

**4. Компактные автономные модели (Vosk).** Vosk построен на другой схеме: небольшая акустическая модель предсказывает звуки речи по кадрам, а словарь произношений и языковая модель (вероятности последовательностей слов) выбирают текст. Маленькая русская модель занимает 45 МБ, работает на процессоре в реальном времени, распознает поток (текст появляется, пока человек говорит), не нужен интернет. Слова – только из ее словаря, числа пишутся словами, знаков препинания нет. Набор слов можно ограничить списком команд – тогда ошибок на командах меньше.

**5. Метрики.** **Расстояние редактирования** (расстояние Левенштейна) – наименьшее число замен $S$, удалений $D$ и вставок $I$, которыми одна последовательность превращается в другую. Его считают динамическим программированием по таблице $d_{i,j}$ – расстояние между первыми $i$ элементами эталона и первыми $j$ элементами гипотезы:
$$d_{i,0} = i, \quad d_{0,j} = j, \quad d_{i,j} = \min\big(d_{i-1,j-1} + [r_i \ne h_j],\ d_{i-1,j} + 1,\ d_{i,j-1} + 1\big).$$
**Доля ошибок по словам** $\mathrm{WER} = (S + D + I) / N$, где $N$ – число слов эталона; **доля ошибок по символам** CER – то же по символам (пробелы тоже символы). WER может быть больше 1, если модель вставила много лишних слов: эталон «стоп», гипотеза «стоп стоп стоп» – WER 2. Перед сравнением тексты **нормализуют**: нижний регистр, «ё» → «е», без знаков препинания. Для набора фраз ошибки и слова складывают по всем фразам, а не усредняют доли.

Пример: эталон «построй маршрут до улицы Весенней, дом двадцать пять», гипотеза «построй маршрут до улица весенней дом 25»: замена «улицы» → «улица», удаление «двадцать», замена «пять» → «25»; WER = 3 / 8 = 0,375. Число «25» записано цифрами – смысл верный, а WER считает ошибкой; такие ошибки исправляет нормализация текста (Т2.Б2.4).

**6. Скорость.** **Коэффициент реального времени** (англ. real-time factor, RTF) = время распознавания / длительность записи. RTF = 0,1 – запись в 10 секунд распознается за 1 секунду; RTF больше 1 – модель не успевает за речью. Для голосового помощника нужен RTF заметно меньше 1 на том компьютере, где он будет работать.

**7. Что ломает распознавание:** шум (двигатель, дорога, музыка), акцент и дефекты речи, далекий микрофон, редкие слова и названия улиц (их мало в обучающих данных), числа (запись цифрами или словами), обрезанное начало или конец фразы.

**Нужное из библиотек.**
- `torch.nn.functional.ctc_loss(log_probs, targets, input_lengths, target_lengths, blank=0, reduction='sum')` – минус логарифм вероятности текста; `log_probs` размера (кадры, пакет, алфавит);
- `jiwer.wer(эталон, гипотеза)`, `jiwer.cer(эталон, гипотеза)` – для сверки; обе принимают строки или списки строк;
- Vosk: `Model(папка)`, `KaldiRecognizer(model, 16000)`, `rec.AcceptWaveform(байты)`, `json.loads(rec.FinalResult())['text']`; звук подается 16-битными целыми числами;
- Whisper: `WhisperProcessor.from_pretrained(имя)`, `WhisperForConditionalGeneration.from_pretrained(имя)`, `processor(x, sampling_rate=16000, return_tensors='pt').input_features`, `model.generate(input_features=..., language='ru', task='transcribe')`, `processor.batch_decode(ids, skip_special_tokens=True)`;
- `time.perf_counter()` – точное время для RTF.

In [ ]:
# Ячейка подготовки: режим, источник данных, общие функции
MODE = 'легкий'          # 'легкий' – компьютер в аудитории; 'полный' – Colab с видеокартой
DATA_SOURCE = 'запасной' # 'группа' – фразы группы в data/phrases; 'запасной' – фразы, озвученные Piper

import csv
import json
import platform
import re
import subprocess
import sys
import time
import urllib.request
import wave
import zipfile
from collections import Counter
from itertools import product
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display
import matplotlib.pyplot as plt
from IPython.display import Audio, Image, display

assert MODE in ('легкий', 'полный') and DATA_SOURCE in ('группа', 'запасной')
DEVICE = 'cuda' if (MODE == 'полный' and torch.cuda.is_available()) else 'cpu'
print('Python', platform.python_version(), '| PyTorch', torch.__version__, '| режим:', MODE, '| данные:', DATA_SOURCE, '| устройство:', DEVICE)

def ensure(module, package, version):
    """Ставит библиотеку фиксированной версии, только если ее нет."""
    try:
        __import__(module)
    except ImportError:
        print(f'устанавливаю {package}=={version} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--retries', '1', '--timeout', '10', f'{package}=={version}'], check=False)

ensure('jiwer', 'jiwer', '3.1.0')
ensure('vosk', 'vosk', '0.3.45')
ensure('transformers', 'transformers', '4.51.3')

SR = 16000
RANDOM_STATE = 42
NAVY, BLUE, DEEP, GRAY, LIGHT = '#0F1B2D', '#4C9BFF', '#1F63C9', '#9AA0A8', '#EAF2FF'
DATA, MODELS = Path('data'), Path('models')

def model_path(name):
    """Папка models/<имя>, если модель сохранена заранее (prepare_data.py), иначе имя на Hugging Face."""
    local = MODELS / name.split('/')[-1]
    return str(local) if (local / 'config.json').exists() else name

def show_fig(fig, name):
    fig.tight_layout(); fig.savefig(name, dpi=110); plt.close(fig)
    display(Image(name))

def read_wav(path):
    """WAV в массив float32 от −1 до 1, моно, 16 000 Гц."""
    with wave.open(str(path), 'rb') as f:
        sr, ch, width, n = f.getframerate(), f.getnchannels(), f.getsampwidth(), f.getnframes()
        raw = f.readframes(n)
    dtype = {1: np.uint8, 2: np.int16, 4: np.int32}[width]
    x = np.frombuffer(raw, dtype=dtype).astype(np.float32)
    x = (x - 128) / 128 if width == 1 else x / float(2 ** (8 * width - 1))
    x = x.reshape(-1, ch).mean(axis=1)
    if sr != SR:
        x = np.interp(np.arange(int(len(x) * SR / sr)) * sr / SR, np.arange(len(x)), x).astype(np.float32)
    return x

def engine_noise(n, rng, rpm=2400):
    """Шум двигателя из части 1З: гармоники частоты вспышек и низкочастотный гул."""
    t = np.arange(n) / SR; f0 = rpm / 60 * 2
    x = sum(np.sin(2 * np.pi * k * f0 * t + rng.uniform(0, 2 * np.pi)) / k for k in range(1, 8))
    rumble = np.cumsum(rng.normal(size=n)); rumble -= np.convolve(rumble, np.ones(400) / 400, mode='same')
    x = x / x.std() + 0.7 * rumble / rumble.std(); return x / x.std()

def add_noise(x, snr_db, rng):
    noise = engine_noise(len(x), rng)
    k = np.sqrt(np.mean(x ** 2) / (np.mean(noise ** 2) * 10 ** (snr_db / 10)))
    return (x + k * noise).astype(np.float32)

### Задание Р1. Декодирование CTC

Допишите по п. 2 теории:
- `ctc_collapse(path, blank='-')` – склеить подряд идущие одинаковые символы и удалить пустые (символ добавляется, если он не пустой и отличается от предыдущего символа пути);
- `ctc_greedy(probs, alphabet)` – самый вероятный символ в каждом кадре (`argmax` по строке), затем `ctc_collapse`;
- `ctc_text_prob(probs, alphabet, text)` – вероятность текста полным перебором всех путей (`itertools.product`) – только для крошечных примеров: сумма произведений вероятностей тех путей, которые склеиваются в `text`.

Затем сверим перебор с `F.ctc_loss`: $e^{-\text{потери}}$ должно совпасть с суммой.

In [ ]:
def ctc_collapse(path, blank='-'):
    out, prev = [], None
    for ch in path:
        if ...:                                     # <-- не повтор и не пустой
            out.append(ch)
        prev = ch
    return ''.join(out)

def ctc_greedy(probs, alphabet):
    path = ''.join(alphabet[i] for i in ...)        # <-- самый вероятный символ кадра
    return ...

def ctc_text_prob(probs, alphabet, text):
    total = 0.0
    for path in product(range(len(alphabet)), repeat=len(probs)):
        if ...:
            total += ...                            # <-- вероятность пути
    return total


print(ctc_collapse('--сс-т-оо--п-'), ctc_collapse('кк-а-сс-с-аа'), ctc_collapse('кассса'), repr(ctc_collapse('-----')), '  ожидается стоп касса каса \'\'')

ALPHABET = ['-', 'с', 'т', 'о', 'п']
plan, conf = '-сс-ттооо-п-', [0.7, 0.6, 0.55, 0.5, 0.65, 0.45, 0.6, 0.7, 0.5, 0.6, 0.8, 0.75]
PROBS = np.array([[p if a == c else (1 - p) / 4 for a in ALPHABET] for c, p in zip(plan, conf)])   # 12 кадров × 5 символов
print('жадный путь:', ''.join(ALPHABET[i] for i in PROBS.argmax(1)), '→', ctc_greedy(PROBS, ALPHABET), '  ожидается -сс-ттооо-п- → стоп')

A2 = ['-', 'а', 'б']
Q = np.array([[0.5, 0.3, 0.2], [0.4, 0.3, 0.3], [0.3, 0.2, 0.5]])
p_ab = ctc_text_prob(Q, A2, 'аб')
loss = F.ctc_loss(torch.log(torch.tensor(Q))[:, None, :], torch.tensor([[1, 2]]), torch.tensor([3]), torch.tensor([2]), blank=0, reduction='sum')
print(f'перебор: {p_ab:.4f}, ctc_loss: {torch.exp(-loss).item():.4f}', '  ожидается 0.2520 0.2520 (пути -аб, а-б, ааб, аб-, абб)')
G = np.array([[0.6, 0.4], [0.6, 0.4]])
print('жадно:', repr(ctc_greedy(G, ['-', 'а'])), f'| P(пусто) = {ctc_text_prob(G, ["-", "а"], ""):.2f}, P(«а») = {ctc_text_prob(G, ["-", "а"], "а"):.2f}',
      "  ожидается '' | 0.36, 0.64 – жадный выбор нашел не самый вероятный текст")

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.imshow(PROBS.T, cmap='Blues', aspect='auto')
ax.set_yticks(range(5), ALPHABET); ax.set_xlabel('кадр'); ax.set_title('вероятности символов по кадрам')
for t in range(len(PROBS)):
    ax.text(t, PROBS[t].argmax(), f'{PROBS[t].max():.2f}', ha='center', va='center', color='white', fontsize=8)
show_fig(fig, 'r1_ctc.png')

**Вопрос Р1.** а) Декодируйте вручную пути `-пп-рр-ии--вв-ее-тт` и `дд-оо-м--мм` и объясните, зачем в CTC пустой символ. б) Почему вероятность текста – сумма по многим путям, а не вероятность одного пути? в) Почему CTC-модели не нужна разметка, какой кадр какой букве соответствует?

*Ваш ответ:*

### Задание Р2. Расстояние редактирования, WER и CER

Допишите по п. 5 теории:
- `align(ref, hyp)` – таблица $d$ динамического программирования для двух списков и обратный проход от правого нижнего угла: если $d_{i,j} = d_{i-1,j-1} + [r_i \ne h_j]$ – совпадение `'='` или замена `'S'`, иначе если $d_{i,j} = d_{i-1,j} + 1$ – удаление `'D'`, иначе вставка `'I'`. Порядок проверок важен: при равных вариантах так получится одно и то же выравнивание у всех. Вернуть расстояние и список операций (операция, слово эталона, слово гипотезы) от начала к концу;
- `wer(refs, hyps)` и `cer(refs, hyps)` – по списку пар фраз: нормализовать (`normalize`, готова), сложить расстояния и длины эталонов по всем фразам и разделить.

In [ ]:
def normalize(text):
    """Нижний регистр, «ё» → «е», без знаков препинания, одиночные пробелы."""
    text = text.lower().replace('ё', 'е')
    return ' '.join(re.sub(r'[^\w\s]', ' ', text).split())

def align(ref, hyp):
    n, m = len(ref), len(hyp)
    d = np.zeros((n + 1, m + 1), dtype=int)
    d[:, 0], d[0, :] = np.arange(n + 1), np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i, j] = ...                           # <-- замена, удаление, вставка
    ops, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i, j] == d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(('=' if ref[i - 1] == hyp[j - 1] else 'S', ref[i - 1], hyp[j - 1])); i, j = i - 1, j - 1
        elif ...:                                   # <-- удаление
            ops.append(('D', ref[i - 1], None)); i -= 1
        else:
            ...                                     # <-- вставка
    return int(d[n, m]), ops[::-1]

def wer(refs, hyps):
    errors = sum(... for r, h in zip(refs, hyps))
    words = sum(... for r in refs)
    return errors / words

def cer(refs, hyps):
    errors = sum(... for r, h in zip(refs, hyps))
    return errors / sum(... for r in refs)


import jiwer
print(normalize('Ёлка, ёж!'), '  ожидается елка еж')
examples = [('а б в г', 'а х в г д'), ('стоп', 'стоп стоп стоп'),
            ('Построй маршрут до улицы Весенней, дом двадцать пять.', 'построй маршрут до улица весенней дом 25')]
show_ops = lambda r, h: ', '.join(f'{op} {a or ""}→{b or ""}' for op, a, b in align(normalize(r).split(), normalize(h).split())[1] if op != '=')
for r, h in examples:
    print(f'WER {wer([r], [h]):.4f}, CER {cer([r], [h]):.4f} |', show_ops(r, h))
print('  ожидается: WER 0.5000, CER 0.4286 | S б→х, I →д')
print('             WER 2.0000, CER 2.5000 | I →стоп, I →стоп')
print('             WER 0.3750, CER 0.2745 | S улицы→улица, D двадцать→, S пять→25')
refs, hyps = [normalize(r) for r, _ in examples], [normalize(h) for _, h in examples]
print('совпадает с jiwer:', np.isclose(wer(refs, hyps), jiwer.wer(refs, hyps)), np.isclose(cer(refs, hyps), jiwer.cer(refs, hyps)), '  ожидается True True')
print(f'по трем фразам вместе: WER {wer(refs, hyps):.4f}', '  ожидается 0.5385 (7 ошибок на 13 слов)')

**Вопрос Р2.** а) Посчитайте вручную WER для эталона «включи музыку погромче» и гипотезы «включи музыка громче пожалуйста». б) Почему WER может быть больше 1, а точность классификатора – нет? в) Почему для набора фраз ошибки складывают, а не усредняют доли по фразам?

*Ваш ответ:*

### Записи для распознавания

Готовая ячейка загружает фразы: записи группы из `data/phrases` или запасные фразы, озвученные Piper, из `data/reserve_tts` (если их нет, ячейка ставит `piper-tts`, скачивает голоса и озвучивает фразы – около минуты). К каждой записи добавляется версия с шумом двигателя.

In [ ]:
# Готовая ячейка: фразы водителя, озвучивание Piper, версии с шумом
PHRASES = [
    'построй маршрут до улицы весенней дом двадцать пять', 'сколько километров до кемерово',
    'поверни направо через триста метров', 'установи температуру двадцать два градуса',
    'позвони саше', 'включи радио сто четыре и восемь', 'найди заправку на кузнецком проспекте',
    'какое давление в передних шинах', 'едем на проспект ленина дом сорок семь', 'сделай потеплее на два градуса',
    'включи следующую песню', 'сколько бензина осталось в баке', 'остановись у дома номер девятнадцать',
    'построй маршрут до притомского проспекта', 'через сколько минут мы приедем', 'выключи кондиционер',
    'позвони маме на мобильный', 'какая скорость разрешена на улице кирова', 'открой окно водителя',
    'едем в аэропорт имени леонова']
VOICES = ['irina', 'denis']
TTS_DIR = DATA / 'reserve_tts'

def build_reserve_tts():
    """Озвучивание фраз синтезатором Piper двумя русскими голосами."""
    ensure('piper', 'piper-tts', '1.3.0')
    from piper import PiperVoice
    base = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/ru/ru_RU/{v}/medium/ru_RU-{v}-medium.onnx'
    for v in VOICES:
        onnx = MODELS / f'ru_RU-{v}-medium.onnx'
        for suffix in ('', '.json'):
            if not Path(str(onnx) + suffix).exists():
                MODELS.mkdir(exist_ok=True); urllib.request.urlretrieve(base.format(v=v) + suffix, str(onnx) + suffix)
        voice = PiperVoice.load(str(onnx))
        (TTS_DIR / v).mkdir(parents=True, exist_ok=True)
        for k, text in enumerate(PHRASES):
            with wave.open(str(TTS_DIR / v / f'{k:02d}.wav'), 'wb') as f:
                if hasattr(voice, 'synthesize_wav'):
                    voice.synthesize_wav(text, f)
                else:   # piper-tts до версии 1.3
                    voice.synthesize(text, f)
    print('озвучено:', len(VOICES) * len(PHRASES), 'фраз')

def load_items():
    """Список записей: словари с сигналом, эталоном, диктором и условием (чисто или шум)."""
    items = []
    if DATA_SOURCE == 'группа':
        for wav in sorted((DATA / 'phrases').glob('*/*.wav')):
            txt = wav.with_suffix('.txt')
            if txt.exists():
                items.append(dict(x=read_wav(wav), ref=normalize(txt.read_text(encoding='utf-8')), speaker=wav.parent.name, cond='чисто'))
    else:
        if not all((TTS_DIR / v / f'{len(PHRASES) - 1:02d}.wav').exists() for v in VOICES):
            build_reserve_tts()
        voices, n = (VOICES[:1], 10) if MODE == 'легкий' else (VOICES, len(PHRASES))
        for v in voices:
            for k in range(n):
                items.append(dict(x=read_wav(TTS_DIR / v / f'{k:02d}.wav'), ref=PHRASES[k], speaker=v, cond='чисто'))
    assert items, 'записей не найдено: проверьте data/phrases или выберите DATA_SOURCE = "запасной"'
    rng = np.random.default_rng(RANDOM_STATE)
    items += [dict(it, x=add_noise(it['x'], 5, rng), cond='шум') for it in list(items)]
    return items

ITEMS = load_items()
dur = sum(len(it['x']) for it in ITEMS) / SR
print(f'{len(ITEMS)} записей ({sum(it["cond"] == "чисто" for it in ITEMS)} чистых и столько же с шумом), всего {dur:.0f} с звука, дикторов: {len(set(it["speaker"] for it in ITEMS))}')
print('  ожидается для запасного набора:', '20 записей (10 фраз голосом irina и они же с шумом)' if MODE == 'легкий' else '80 записей (20 фраз двумя голосами и они же с шумом)')
for it in (ITEMS[0], ITEMS[len(ITEMS) // 2]):
    print(it['cond'], '|', it['ref']); display(Audio(it['x'], rate=SR))

### Задание Р3. Vosk на процессоре

Готовый код скачивает модель `vosk-model-small-ru-0.22` (45 МБ) в папку `models`, если ее нет. Допишите `vosk_transcribe(x)`: перевести сигнал в 16-битные целые числа (`np.clip(x, -1, 1) * 32767`, тип `np.int16`, затем `.tobytes()`), создать распознаватель `KaldiRecognizer(VOSK_MODEL, SR)`, подать байты кусками по 8000 байт через `AcceptWaveform` (так звук приходит с микрофона) и взять текст из `FinalResult()`.

In [ ]:
from vosk import KaldiRecognizer, Model, SetLogLevel
SetLogLevel(-1)
VOSK_DIR = MODELS / 'vosk-model-small-ru-0.22'
if not VOSK_DIR.exists():
    MODELS.mkdir(exist_ok=True)
    print('скачиваю модель Vosk (45 МБ) ...')
    urllib.request.urlretrieve('https://alphacephei.com/vosk/models/vosk-model-small-ru-0.22.zip', MODELS / 'vosk.zip')
    zipfile.ZipFile(MODELS / 'vosk.zip').extractall(MODELS)
VOSK_MODEL = Model(str(VOSK_DIR))

def vosk_transcribe(x, grammar=None):
    pcm = ...                                       # <-- 16-битные целые в байтах
    rec = KaldiRecognizer(VOSK_MODEL, SR) if grammar is None else KaldiRecognizer(VOSK_MODEL, SR, json.dumps(grammar, ensure_ascii=False))
    for s in range(0, len(pcm), 8000):
        ...                                         # <-- подаем звук кусками
    return ...


t0 = time.perf_counter()
vosk_hyps = [vosk_transcribe(it['x']) for it in ITEMS]
print(f'Vosk: {time.perf_counter() - t0:.1f} с на {dur:.0f} с звука')
for it, h in list(zip(ITEMS, vosk_hyps))[:5]:
    print(f'  эталон:   {it["ref"]}\n  гипотеза: {h}')
for cond in ('чисто', 'шум'):
    idx = [k for k, it in enumerate(ITEMS) if it['cond'] == cond]
    print(f'{cond}: WER {wer([ITEMS[k]["ref"] for k in idx], [vosk_hyps[k] for k in idx]):.3f}')
print('ожидается: тексты строчными буквами, числа словами, без знаков препинания; с шумом WER заметно выше')

**Вопрос Р3.** а) Какие слова Vosk распознает хуже всего? б) Почему Vosk пишет числа словами, а знаков препинания у него нет? в) Зачем подавать звук кусками, если вся запись уже есть?

*Ваш ответ:*

### Задание Р4. Whisper: качество, скорость, размер

Допишите:
- `whisper_transcribe(x, model, processor)` – признаки `processor(x, sampling_rate=SR, return_tensors='pt').input_features` (перенести на устройство и в формат модели), `model.generate(input_features=..., language='ru', task='transcribe', max_new_tokens=128)` и текст `processor.batch_decode(..., skip_special_tokens=True)[0]`;
- `evaluate(transcribe, items)` – распознать все записи, засечь время `time.perf_counter()` для каждой, вернуть гипотезы, WER, CER и коэффициент реального времени (суммарное время распознавания / суммарная длительность записей).

Готовая ячейка собирает таблицу «модель – WER (чисто и шум) – CER – коэффициент реального времени – параметры – размер».

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import transformers
transformers.logging.set_verbosity_error()

def whisper_transcribe(x, model, processor):
    feats = ...                                     # <-- (1, 80, 3000)
    feats = feats.to(model.device, model.dtype)
    with torch.no_grad():
        ids = ...
    return ...

def evaluate(transcribe, items):
    hyps, spent = [], 0.0
    for it in items:
        t0 = time.perf_counter()
        hyps.append(transcribe(it['x']))
        spent += ...
    audio = sum(len(it['x']) for it in items) / SR
    refs = [it['ref'] for it in items]
    return hyps, ..., ..., ...                      # <-- гипотезы, WER, CER, RTF


processor = WhisperProcessor.from_pretrained(model_path('openai/whisper-tiny'))
feats = processor(ITEMS[0]['x'], sampling_rate=SR, return_tensors='pt').input_features
print('вход Whisper:', tuple(feats.shape), '  ожидается (1, 80, 3000): 80 мел-полос, 30 секунд по 10 мс')
tiny = WhisperForConditionalGeneration.from_pretrained(model_path('openai/whisper-tiny')).to(DEVICE)
ids = tiny.generate(input_features=feats.to(DEVICE), language='ru', task='transcribe', max_new_tokens=64)
print('со служебными токенами:', processor.batch_decode(ids, skip_special_tokens=False)[0])
print('  ожидается: начало <|startoftranscript|><|ru|><|transcribe|><|notimestamps|>, в конце <|endoftext|>')
print('параметров tiny:', sum(p.numel() for p in tiny.parameters()), '  ожидается 37760640')

In [ ]:
# Готовая ячейка: таблица моделей
names = ['openai/whisper-tiny', 'openai/whisper-base'] + (['openai/whisper-small'] if MODE == 'полный' else [])
clean = [it for it in ITEMS if it['cond'] == 'чисто']
noisy = [it for it in ITEMS if it['cond'] == 'шум']
HYPS, rows = {}, []

def add_row(name, transcribe, n_params, size_mb):
    h_clean, w_clean, c_clean, rtf = evaluate(transcribe, clean)
    h_noisy, w_noisy, _, _ = evaluate(transcribe, noisy)
    HYPS[name] = {'чисто': h_clean, 'шум': h_noisy}
    rows.append((name, w_clean, w_noisy, c_clean, rtf, n_params, size_mb))
    print(f'{name}: готово')

vosk_size = sum(f.stat().st_size for f in VOSK_DIR.rglob('*') if f.is_file()) / 1e6
add_row('vosk-small-ru', vosk_transcribe, None, vosk_size)
for name in names:
    proc = WhisperProcessor.from_pretrained(model_path(name))
    dtype = torch.float16 if DEVICE == 'cuda' else torch.float32
    new_api = tuple(int(v) for v in re.findall(r'\d+', transformers.__version__)[:2]) >= (4, 56)   # в новых версиях аргумент называется dtype
    model = WhisperForConditionalGeneration.from_pretrained(model_path(name), **{'dtype' if new_api else 'torch_dtype': dtype}).to(DEVICE).eval()
    n = sum(p.numel() for p in model.parameters())
    add_row(name.split('/')[-1], lambda x, m=model, p=proc: whisper_transcribe(x, m, p), n, n * 4 / 1e6)
    del model

print(f'\n{"модель":<15} {"WER чисто":>10} {"WER шум":>8} {"CER":>6} {"RTF":>6} {"параметров":>12} {"размер, МБ":>11}')
for name, wc, wn, c, rtf, n, mb in rows:
    print(f'{name:<15} {wc:10.3f} {wn:8.3f} {c:6.3f} {rtf:6.3f} {n if n else "–":>12} {mb:11.0f}')
print('ожидается: параметров 37760640, 72593920, 241734912; размер в 32 битах около 151, 290, 967 МБ, у Vosk – размер папки модели на диске, несколько десятков МБ;',
      'Whisper tiny и base на процессоре медленнее Vosk; WER Whisper растет из-за чисел цифрами и знаков препинания до нормализации – их убирает только normalize; с шумом WER у всех выше')

**Вопрос Р4.** а) Какая модель точнее на чистых записях и какая на записях с шумом? б) Какая модель быстрее всех и во сколько раз быстрее реального времени? Что будет с коэффициентом реального времени на компьютере в машине? в) Найдите в гипотезах Whisper числа, записанные цифрами. Почему WER засчитывает их как ошибки, хотя смысл верный?

*Ваш ответ:*

### Задание Р5. Ошибки по типам

Разберем, какие ошибки делает модель. Допишите:
- `category(word)` – тип слова эталона: `'число'`, если слово есть в списке `NUMBER_WORDS` или содержит цифру; `'название'`, если оно есть в списке `NAMES`; иначе `'другое'`;
- `error_table(refs, hyps)` – выровнять каждую пару фраз функцией `align` по словам и посчитать ошибки: для замены и удаления – по типу слова эталона, для вставки – тип `'вставка'`; вернуть `Counter` и список строк (операция, слово эталона, слово гипотезы, тип) для файла.

Таблица ошибок сохраняется в `errors.csv`: она понадобится на занятии Т2.Б2.4, где мы будем исправлять числа и названия.

In [ ]:
NUMBER_WORDS = {'ноль', 'один', 'одна', 'два', 'две', 'три', 'четыре', 'пять', 'шесть', 'семь', 'восемь', 'девять', 'десять',
                'одиннадцать', 'двенадцать', 'тринадцать', 'четырнадцать', 'пятнадцать', 'шестнадцать', 'семнадцать',
                'восемнадцать', 'девятнадцать', 'двадцать', 'тридцать', 'сорок', 'пятьдесят', 'шестьдесят', 'семьдесят',
                'восемьдесят', 'девяносто', 'сто', 'двести', 'триста', 'четыреста', 'пятьсот', 'тысяча'}
NAMES = {'весенней', 'кемерово', 'саше', 'кузнецком', 'ленина', 'притомского', 'кирова', 'леонова'}

def category(word):
    if ...:
        return 'число'
    return ...

def error_table(refs, hyps):
    counts, rows = Counter(), []
    for r, h in zip(refs, hyps):
        for op, rw, hw in align(normalize(r).split(), normalize(h).split())[1]:
            if op in ('S', 'D'):
                kind = ...
            elif op == 'I':
                kind = ...
            else:
                continue
            counts[kind] += 1
            rows.append((op, rw, hw, kind))
    return counts, rows


demo_refs = ['построй маршрут до улицы весенней дом двадцать пять', 'позвони саше', 'выключи кондиционер']
demo_hyps = ['построй маршрут до улица весенний дом 25', 'позвонить саша', 'выключи кондиционер пожалуйста']
print(dict(error_table(demo_refs, demo_hyps)[0]), "  ожидается {'другое': 2, 'название': 2, 'число': 2, 'вставка': 1}")

refs_all = {cond: [it['ref'] for it in ITEMS if it['cond'] == cond] for cond in ('чисто', 'шум')}
with open('errors.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['модель', 'условие', 'операция', 'эталон', 'гипотеза', 'тип'])
    print(f'\n{"модель":<15} {"условие":<7} ' + ' '.join(f'{k:>9}' for k in ('число', 'название', 'другое', 'вставка')))
    for name, by_cond in HYPS.items():
        for cond, hyps in by_cond.items():
            counts, rows_e = error_table(refs_all[cond], hyps)
            print(f'{name:<15} {cond:<7} ' + ' '.join(f'{counts[k]:9d}' for k in ('число', 'название', 'другое', 'вставка')))
            writer.writerows([(name, cond, *row) for row in rows_e])
n_words = {k: sum(category(w) == k for r in refs_all['чисто'] for w in r.split()) for k in ('число', 'название', 'другое')}
print('слов в эталонах (чистые записи):', n_words)
print('ожидается: у Whisper много ошибок в числах (цифры вместо слов), у всех моделей – в названиях улиц; с шумом растут все типы; таблица сохранена в errors.csv')

**Вопрос Р5.** а) Ошибок какого типа больше всего относительно числа таких слов в эталонах? б) Какие ошибки можно исправить после распознавания, не меняя модель, и как? в) Какие ошибки исправить после распознавания нельзя?

*Ваш ответ:*

### Задание Р6. Какую модель взять для помощника

Голосовой помощник водителя работает на компьютере в машине: процессор уровня ноутбука из аудитории, видеокарты нет, интернет есть не всегда, в салоне шум двигателя и дороги. Помощник должен понимать команды и адреса, отвечать без заметной задержки.

**Вопрос Р6.** Выберите модель распознавания для помощника по своей таблице из Р4 и Р5 и обоснуйте выбор по пяти признакам: качество на чистых записях, качество с шумом, коэффициент реального времени, размер, работа без интернета. Что вы сделаете, чтобы уменьшить слабые стороны выбранной модели?

*Ваш ответ:*

### Задания повышенной сложности (к части 2Р)

1. **Лучевой поиск для CTC.** Напишите лучевой поиск по префиксам: для каждого кадра храните до `beam` лучших префиксов текста с двумя вероятностями – путей, кончающихся пустым символом, и путей, кончающихся последней буквой префикса; при переходе к новому кадру складывайте вероятности путей, дающих один и тот же префикс. На матрице `G` из Р1 лучевой поиск с шириной 2 должен найти текст «а» с вероятностью 0,64, а жадное декодирование – пустой текст. Проверьте на `PROBS`, что результат «стоп» не изменился.
2. **Ограничение словаря Vosk.** Передайте в `vosk_transcribe` грамматику – список фраз `['вперед', 'стоп', 'налево', 'направо', 'быстрее', 'медленнее', '[unk]']` – и распознайте записи команд из части 1З (папка `data/commands` или `data/reserve` части 1З). Сравните долю верно распознанных команд со свободным распознаванием и с сетью SimpleCNN из части 1З. Почему ограничение словаря так сильно помогает?
3. **Отметки времени.** Получите от Whisper отметки времени по словам (`return_timestamps='word'` в конвейере `pipeline('automatic-speech-recognition', ...)`) и нарисуйте мел-спектрограмму фразы с границами слов.

In [ ]:
# Место для заданий повышенной сложности